In [ ]:



import subprocess, os

TMP     = '/kaggle/tmp'       
WORKING = '/kaggle/working'   
os.makedirs(TMP,     exist_ok=True)
os.makedirs(WORKING, exist_ok=True)

def download_if_missing(url, dest, label):
    if os.path.exists(dest) and os.path.getsize(dest) > 1_000_000:
        print(f'{label}: already downloaded → {dest}')
        return
    print(f'{label}: starting download...')
    result = subprocess.run(
        ['curl', '-L', '--retry', '3', url, '-o', dest, '--progress-bar'],
        text=True
    )
    size = os.path.getsize(dest)
    print(f'{label}: done. {size/1e9:.2f} GB')

download_if_missing(
    'https://zenodo.org/records/6390798/files/embryo_dataset_annotations.tar.gz',
    f'{TMP}/embryo_dataset_annotations.tar.gz',
    'Annotations'
)
print('Starting 12.1 GB image download... (10-30 min)')
download_if_missing(
    'https://zenodo.org/records/6390798/files/embryo_dataset.tar.gz',
    f'{TMP}/embryo_dataset.tar.gz',
    'Images'
)
print('All downloads complete.')

Annotations: starting download...


######################################################################## 100.0%


Annotations: done. 0.00 GB
Starting 12.1 GB image download... (10-30 min)
Images: starting download...


#######################################################################  100.0%

Images: done. 12.15 GB
All downloads complete.


######################################################################## 100.0%


In [ ]:
# Extract tarballs to /kaggle/tmp ──────────────────────────────


import tarfile, os

TMP     = '/kaggle/tmp'
WORKING = '/kaggle/working'

def extract_if_needed(tar_path, extract_to, label):
    if os.path.exists(extract_to) and os.listdir(extract_to):
        print(f'{label}: already extracted → {extract_to}')
        return
    os.makedirs(extract_to, exist_ok=True)
    print(f'{label}: extracting {tar_path} ...')
    with tarfile.open(tar_path, 'r:gz') as tar:
        tar.extractall(extract_to)
    print(f'{label}: done.')

extract_if_needed(f'{TMP}/embryo_dataset_annotations.tar.gz', f'{TMP}/annotations', 'Annotations')
extract_if_needed(f'{TMP}/embryo_dataset.tar.gz',             f'{TMP}/embryo_dataset', 'Images')

def resolve_root(path):
    entries = os.listdir(path)
    if len(entries) == 1 and os.path.isdir(os.path.join(path, entries[0])):
        return os.path.join(path, entries[0])
    return path

IMAGE_ROOT      = resolve_root(f'{TMP}/embryo_dataset')
ANNOTATION_ROOT = resolve_root(f'{TMP}/annotations')

assert os.path.exists(IMAGE_ROOT),      f'Not found: {IMAGE_ROOT}'
assert os.path.exists(ANNOTATION_ROOT), f'Not found: {ANNOTATION_ROOT}'

# Show disk usage
import shutil
total, used, free = shutil.disk_usage(TMP)
print(f'\nImage root      : {IMAGE_ROOT}')
print(f'Annotation root : {ANNOTATION_ROOT}')
print(f'Image folders   : {len(os.listdir(IMAGE_ROOT))}')
print(f'Annotation CSVs : {len([f for f in os.listdir(ANNOTATION_ROOT) if f.endswith(".csv")])}')
print(f'\nDisk ({TMP}): {used/1e9:.1f} GB used / {total/1e9:.1f} GB total  ({free/1e9:.1f} GB free)')

Annotations: extracting /kaggle/tmp/embryo_dataset_annotations.tar.gz ...
Annotations: done.
Images: extracting /kaggle/tmp/embryo_dataset.tar.gz ...


/tmp/ipykernel_55/919527289.py:16: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(extract_to)


Images: done.

Image root      : /kaggle/tmp/embryo_dataset/embryo_dataset
Annotation root : /kaggle/tmp/annotations/embryo_dataset_annotations
Image folders   : 704
Annotation CSVs : 704

Disk (/kaggle/tmp): 7371.8 GB used / 8656.9 GB total  (1285.1 GB free)


In [ ]:

import os, random
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from PIL import Image, ImageFile
from PIL import UnidentifiedImageError
ImageFile.LOAD_TRUNCATED_IMAGES = True

from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.metrics import f1_score, classification_report

# torchinfo — install if missing (Kaggle usually has it)
try:
    from torchinfo import summary
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'torchinfo', '-q'], check=True)
    from torchinfo import summary

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
if device.type == 'cuda':
    print('GPU :', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:

CONFIG = {
    'mobilenet': {'batch_size': 128, 'img_size': 224},
    'vgg16':     {'batch_size': 64,  'img_size': 224},
    'vgg19':     {'batch_size': 48,  'img_size': 224},
    'inception': {'batch_size': 64,  'img_size': 299},
}

NUM_WORKERS = 4

TMP     = '/kaggle/tmp'       # data (images, tarballs)
WORKING = '/kaggle/working'   # outputs (checkpoints, plots, results CSV)

PHASE_ORDER = [
    'tPB2','tPNa','tPNf','t2','t3','t4','t5',
    't6','t7','t8','t9+','tM','tSB','tB','tEB','tHB'
]
CLASS_TO_IDX = {p: i for i, p in enumerate(PHASE_ORDER)}
IDX_TO_CLASS = {i: p for p, i in CLASS_TO_IDX.items()}
NUM_CLASSES  = len(PHASE_ORDER)

print('Config ready.')
for m, c in CONFIG.items():
    print(f'  {m:10s}: batch={c["batch_size"]}  img_size={c["img_size"]}')


In [ ]:


class EmbryoDataset(Dataset):
    """
    Loads every annotated frame without any capping.
    Each sample: (img_path, label_int, video_id)
    video_id is used for embryo-level train/val/test splitting.
    """
    def __init__(self, image_root, annotation_root, transform=None, debug=True):
        self.class_to_idx = CLASS_TO_IDX
        self.idx_to_class = IDX_TO_CLASS
        self.transform    = transform
        self.samples      = []   # list of (img_path, label, video_id)
        self.class_counts = {p: 0 for p in PHASE_ORDER}

        missing_files = 0
        print(f'\nScanning annotation folder: {annotation_root}')

        for csv_file in sorted(os.listdir(annotation_root)):
            if not csv_file.endswith('.csv'):
                continue

            video_name  = csv_file.replace('_phases.csv', '')
            csv_path    = os.path.join(annotation_root, csv_file)
            folder_path = os.path.join(image_root, video_name)

            if not os.path.exists(folder_path):
                continue

            # Build frame_index → filename map
            run_map = {}
            for f in os.listdir(folder_path):
                if 'RUN' in f:
                    try:
                        run_num = int(f.split('RUN')[-1].split('.')[0])
                        run_map[run_num] = f
                    except ValueError:
                        continue

            df = pd.read_csv(csv_path, header=None)

            for _, row in df.iterrows():
                phase, start, end = row[0], int(row[1]), int(row[2])
                if phase not in self.class_to_idx:
                    continue
                label = self.class_to_idx[phase]
                for frame_idx in range(start, end + 1):
                    if frame_idx in run_map:
                        img_path = os.path.join(folder_path, run_map[frame_idx])
                        self.samples.append((img_path, label, video_name))
                        self.class_counts[phase] += 1
                    else:
                        missing_files += 1

        print(f'\nDATASET SUMMARY')
        print(f'  Total samples  : {len(self.samples):,}')
        print(f'  Total classes  : {len(self.class_to_idx)}')
        print(f'  Missing frames : {missing_files:,}')

        if debug:
            print('\n  Class distribution:')
            for k, v in self.class_counts.items():
                bar = '#' * (v // 5000)
                print(f'    {k:6s}: {v:8,}  {bar}')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        # Loop-based corrupt-image fallback — no recursion risk
        for offset in range(len(self.samples)):
            cur_idx = (idx + offset) % len(self.samples)
            img_path, label, _ = self.samples[cur_idx]
            try:
                image = Image.open(img_path).convert('RGB')
                if self.transform:
                    image = self.transform(image)
                return image, label
            except (OSError, IOError, UnidentifiedImageError):
                continue
        raise RuntimeError(f'No valid images found starting from index {idx}')

In [ ]:


import os

TMP = '/kaggle/tmp'

def resolve_root(path):
    entries = os.listdir(path)
    if len(entries) == 1 and os.path.isdir(os.path.join(path, entries[0])):
        return os.path.join(path, entries[0])
    return path

IMAGE_ROOT      = resolve_root(f'{TMP}/embryo_dataset')
ANNOTATION_ROOT = resolve_root(f'{TMP}/annotations')

base_dataset = EmbryoDataset(IMAGE_ROOT, ANNOTATION_ROOT, transform=None, debug=True)

In [ ]:

# Split is done at the EMBRYO (video) level to prevent data leakage.


from collections import defaultdict

# Map each embryo → set of class labels it contains
embryo_classes = defaultdict(set)
for _, label, vid in base_dataset.samples:
    embryo_classes[vid].add(label)

video_ids = list(embryo_classes.keys())
random.Random(SEED).shuffle(video_ids)

n            = len(video_ids)
train_target = int(0.70 * n)
val_target   = int(0.15 * n)
test_target  = n - train_target - val_target

train_videos, val_videos, test_videos = set(), set(), set()
split_class_counts = {
    'train': defaultdict(int),
    'val':   defaultdict(int),
    'test':  defaultdict(int),
}

def score_split(split_name, vid):
    """Lower = less-represented split; assign here for balance."""
    return sum(split_class_counts[split_name][cls] for cls in embryo_classes[vid])

for vid in video_ids:
    possible = []
    if len(train_videos) < train_target: possible.append('train')
    if len(val_videos)   < val_target:   possible.append('val')
    if len(test_videos)  < test_target:  possible.append('test')
    if not possible:
        possible = ['train']   # overflow goes to train

    best = min(possible, key=lambda s: score_split(s, vid))
    {'train': train_videos, 'val': val_videos, 'test': test_videos}[best].add(vid)
    for cls in embryo_classes[vid]:
        split_class_counts[best][cls] += 1

# Build index lists
train_indices = [i for i, (_, _, v) in enumerate(base_dataset.samples) if v in train_videos]
val_indices   = [i for i, (_, _, v) in enumerate(base_dataset.samples) if v in val_videos]
test_indices  = [i for i, (_, _, v) in enumerate(base_dataset.samples) if v in test_videos]

print(f'Embryo split  : train={len(train_videos)}  val={len(val_videos)}  test={len(test_videos)}')
print(f'\nFrame counts  (FULL dataset — no cap):')
print(f'  Train : {len(train_indices):>10,} frames')
print(f'  Val   : {len(val_indices):>10,} frames')
print(f'  Test  : {len(test_indices):>10,} frames')
print(f'  Total : {len(train_indices)+len(val_indices)+len(test_indices):>10,} frames')

# Leakage check
assert not ({base_dataset.samples[i][2] for i in train_indices} &
            {base_dataset.samples[i][2] for i in val_indices}),   'Train/Val leakage!'
assert not ({base_dataset.samples[i][2] for i in train_indices} &
            {base_dataset.samples[i][2] for i in test_indices}),  'Train/Test leakage!'
assert not ({base_dataset.samples[i][2] for i in val_indices}   &
            {base_dataset.samples[i][2] for i in test_indices}),  'Val/Test leakage!'
print('\nLeakage check passed ✓')

In [ ]:
# ── CELL 8: Class weights (from full train set) ───────────────────────────

train_labels = [base_dataset.samples[i][1] for i in train_indices]
class_counts = np.bincount(train_labels, minlength=NUM_CLASSES)

total   = len(train_labels)
weights = total / (NUM_CLASSES * (class_counts + 1))
weights = np.clip(weights, 0.5, 20.0)

CLASS_WEIGHTS = torch.tensor(weights, dtype=torch.float32).to(device)

print('Class weights (from full training set):')
for i, w in enumerate(CLASS_WEIGHTS):
    print(f'  {IDX_TO_CLASS[i]:6s}: weight={w:.3f}  (n={class_counts[i]:,})')

In [ ]:
# Enhanced Log Loss (ELLoss) 

class ELLoss(nn.Module):
    """
    Enhanced Log Loss.
    1. Focal term     : down-weights easy samples
    2. Class weights  : up-weights rare classes
    3. Ordinal term   : soft differentiable expected-index penalty
    """
    def __init__(self, gamma=1.0, beta=0.05, class_weights=None):
        super().__init__()
        self.gamma = gamma
        self.beta  = beta
        self.register_buffer('class_weights', class_weights)

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.class_weights, reduction='none')

        p      = torch.softmax(logits, dim=1)
        pt     = p.gather(1, targets.view(-1, 1)).squeeze(1)
        focal  = (1 - pt) ** self.gamma

        indices         = torch.arange(logits.size(1), device=logits.device).float()
        expected_idx    = (p * indices).sum(dim=1)
        ordinal_penalty = torch.abs(expected_idx - targets.float())

        return (focal * ce + self.beta * ordinal_penalty).mean()

In [ ]:
#  Transforms + TransformSubset 

class TransformSubset(Dataset):
    """Wraps a Subset and applies a transform at __getitem__ time."""
    def __init__(self, subset, transform=None):
        self.subset    = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        for offset in range(len(self.subset)):
            cur_idx  = (idx + offset) % len(self.subset)
            img_path, label, _ = self.subset.dataset.samples[self.subset.indices[cur_idx]]
            try:
                image = Image.open(img_path).convert('RGB')
                if self.transform:
                    image = self.transform(image)
                return image, label
            except (OSError, IOError, UnidentifiedImageError):
                continue
        raise RuntimeError(f'No valid images found from index {idx}')


def get_transforms(img_size):
    """Train uses augmentation; val/test use only resize + normalize."""
    train_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    eval_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    return train_tf, eval_tf

In [ ]:
#  DataLoaders 

def get_dataloaders(config):
    train_tf, eval_tf = get_transforms(config['img_size'])

    train_set = TransformSubset(Subset(base_dataset, train_indices), train_tf)
    val_set   = TransformSubset(Subset(base_dataset, val_indices),   eval_tf)
    test_set  = TransformSubset(Subset(base_dataset, test_indices),  eval_tf)

    train_loader = DataLoader(
        train_set,
        batch_size=config['batch_size'],
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=(NUM_WORKERS > 0),
    )
    val_loader = DataLoader(
        val_set,
        batch_size=config['batch_size'],
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=(NUM_WORKERS > 0),
    )
    test_loader = DataLoader(
        test_set,
        batch_size=config['batch_size'],
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=(NUM_WORKERS > 0),
    )
    return train_loader, val_loader, test_loader

In [ ]:
#  Model builder 

def freeze_backbone(model, model_name):
    """Freeze everything, unfreeze only the classification head."""
    for p in model.parameters():
        p.requires_grad = False
    if model_name == 'mobilenet':
        for p in model.classifier.parameters():    p.requires_grad = True
    elif model_name in ('vgg16', 'vgg19'):
        for p in model.classifier[6].parameters(): p.requires_grad = True
    elif model_name == 'inception':
        for p in model.fc.parameters():            p.requires_grad = True
        for p in model.AuxLogits.fc.parameters():  p.requires_grad = True
    _print_trainable(model, 'Phase 1 — head only')


def unfreeze_top_blocks(model, model_name):
    
    if model_name == 'vgg16':
        for p in model.features[24:].parameters():  p.requires_grad = True
        for p in model.classifier.parameters():     p.requires_grad = True
    elif model_name == 'vgg19':
        for p in model.features[28:].parameters():  p.requires_grad = True
        for p in model.classifier.parameters():     p.requires_grad = True
    elif model_name == 'mobilenet':
        for p in model.features[12:].parameters():  p.requires_grad = True
        for p in model.classifier.parameters():     p.requires_grad = True
    elif model_name == 'inception':
        for p in model.Mixed_7a.parameters():       p.requires_grad = True
        for p in model.Mixed_7b.parameters():       p.requires_grad = True
        for p in model.Mixed_7c.parameters():       p.requires_grad = True
        for p in model.fc.parameters():             p.requires_grad = True
        for p in model.AuxLogits.parameters():      p.requires_grad = True
    _print_trainable(model, 'Phase 2 — head + last 2 blocks')


def _print_trainable(model, label=''):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f'  {label}: {trainable:,} / {total:,} params trainable ({100*trainable/total:.1f}%)')


def get_model(name, img_size):
    print(f'\nLoading {name.upper()}...')
    if name == 'mobilenet':
        model = models.mobilenet_v2(weights='IMAGENET1K_V1')
        model.classifier[1] = nn.Linear(model.last_channel, NUM_CLASSES)
    elif name == 'vgg16':
        model = models.vgg16(weights='IMAGENET1K_V1')
        model.classifier[6] = nn.Linear(4096, NUM_CLASSES)
    elif name == 'vgg19':
        model = models.vgg19(weights='IMAGENET1K_V1')
        model.classifier[6] = nn.Linear(4096, NUM_CLASSES)
    elif name == 'inception':
        model = models.inception_v3(weights='IMAGENET1K_V1', aux_logits=True)
        model.fc           = nn.Linear(model.fc.in_features, NUM_CLASSES)
        model.AuxLogits.fc = nn.Linear(model.AuxLogits.fc.in_features, NUM_CLASSES)
    else:
        raise ValueError(f'Unknown model: {name}')

    freeze_backbone(model, name)
    return model.to(device)


In [ ]:
# train_one_epoch 
def train_one_epoch(model, loader, optimizer, criterion, model_name, epoch, total_epochs):
    model.train()
    total_loss       = 0.0
    correct          = 0
    total            = 0
    num_batches      = len(loader)
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    log_every        = max(1, num_batches // 10)   # print ~10 times per epoch

    print(f'\n[Epoch {epoch}/{total_epochs}]')

    for batch_idx, (images, labels) in enumerate(loader, 1):
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(images)

        if model_name == 'inception' and isinstance(outputs, tuple):
            main_out, aux_out = outputs
            loss  = criterion(main_out, labels) + 0.4 * criterion(aux_out, labels)
            preds = torch.argmax(main_out, dim=1)
        else:
            if isinstance(outputs, tuple): outputs = outputs[0]
            loss  = criterion(outputs, labels)
            preds = torch.argmax(outputs, dim=1)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=5.0)
        optimizer.step()

        total_loss += loss.item()
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)

        if batch_idx % log_every == 0 or batch_idx == num_batches:
            print(f'  Batch {batch_idx:>6}/{num_batches} | avg loss {total_loss/batch_idx:.4f}')

    train_acc = correct / total
    return total_loss / num_batches, train_acc

In [ ]:
# evaluate 

def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            outputs = model(images)
            if isinstance(outputs, tuple): outputs = outputs[0]

            total_loss += criterion(outputs, labels).item()
            preds       = torch.argmax(outputs, dim=1)
            correct    += (preds == labels).sum().item()
            total      += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = correct / total
    f1  = f1_score(all_labels, all_preds, average='weighted')
    return total_loss / len(loader), acc, f1, all_preds, all_labels

In [ ]:


# Phase 1: head only, higher LR, 3 epochs  — fast stabilisation
# Phase 2: head + last 2 conv blocks, lower LR, 4 epochs 


import csv, os
WORKING = '/kaggle/working'

PHASE_CONFIG = {
    #              p1_epochs  p1_lr   p2_epochs  p2_lr
    'mobilenet': (3,          1e-4,   4,         5e-5),
    'vgg16':     (3,          1e-4,   4,         1e-5),
    'vgg19':     (3,          1e-4,   4,         1e-5),
    'inception': (3,          1e-4,   4,         5e-5),
}


def run_phase(model, model_name, train_loader, val_loader, criterion,
              lr, epochs, phase_label, best_val_loss, save_path, epoch_csv):
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.Adam(trainable_params, lr=lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=lr/100)

    csv_fields = ['phase','epoch','train_loss','train_acc','val_loss','val_acc','val_f1']
    losses_tr, train_accs, losses_val, accs, f1s = [], [], [], [], []

    for ep in range(1, epochs + 1):
        print(f'\n── {phase_label}  Epoch {ep}/{epochs} ──')
        tr_loss, tr_acc                 = train_one_epoch(
            model, train_loader, optimizer, criterion, model_name, ep, epochs
        )
        val_loss, val_acc, val_f1, _, _ = evaluate(model, val_loader, criterion)
        scheduler.step()

        losses_tr.append(tr_loss)
        train_accs.append(tr_acc)
        losses_val.append(val_loss)
        accs.append(val_acc)
        f1s.append(val_f1)

        with open(epoch_csv, 'a', newline='') as f:
            csv.DictWriter(f, fieldnames=csv_fields).writerow({
                'phase':      phase_label, 'epoch': ep,
                'train_loss': round(tr_loss, 6),
                'train_acc':  round(tr_acc,  6),
                'val_loss':   round(val_loss, 6),
                'val_acc':    round(val_acc, 6),
                'val_f1':     round(val_f1, 6),
            })

        tag = '  <- saved' if val_loss < best_val_loss else ''
        print(f'  tr_loss {tr_loss:.4f} | tr_acc {tr_acc:.4f} | val_loss {val_loss:.4f} | val_acc {val_acc:.4f} | F1 {val_f1:.4f}{tag}')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save({
                'epoch':         ep,
                'phase':         phase_label,
                'model_state':   model.state_dict(),
                'best_val_loss': best_val_loss,
            }, save_path)

    return best_val_loss, losses_tr, losses_val, accs, f1s


def train_model(model_name):
    print(f"\n{'='*60}\n  {model_name.upper()}\n{'='*60}")
    config = CONFIG[model_name]
    p1_ep, p1_lr, p2_ep, p2_lr = PHASE_CONFIG[model_name]

    train_loader, val_loader, test_loader = get_dataloaders(config)
    model     = get_model(model_name, config['img_size'])
    criterion = ELLoss(gamma=1.0, beta=0.05, class_weights=CLASS_WEIGHTS)

    save_path = f'{WORKING}/best_{model_name}.pth'
    epoch_csv = f'{WORKING}/{model_name}_epoch_log.csv'
    with open(epoch_csv, 'w', newline='') as f:
        csv.DictWriter(
            f, fieldnames=['phase','epoch','train_loss','train_acc','val_loss','val_acc','val_f1']
        ).writeheader()

    # ── Phase 1: head only ──
    print(f'\nPHASE 1 — head only  ({p1_ep} epochs, lr={p1_lr})')
    best_val_loss, tr1, val1, acc1, f1_1 = run_phase(
        model, model_name, train_loader, val_loader, criterion,
        p1_lr, p1_ep, 'warmup', float('inf'), save_path, epoch_csv
    )

    # ── Phase 2: head + last 2 blocks ──
    print(f'\nPHASE 2 — head + last 2 blocks  ({p2_ep} epochs, lr={p2_lr})')
    unfreeze_top_blocks(model, model_name)
    best_val_loss, tr2, val2, acc2, f1_2 = run_phase(
        model, model_name, train_loader, val_loader, criterion,
        p2_lr, p2_ep, 'finetune', best_val_loss, save_path, epoch_csv
    )

    # ── Test on best checkpoint ──
    print(f'\nLoading best checkpoint (val_loss={best_val_loss:.4f})...')
    ckpt = torch.load(save_path, map_location=device, weights_only=True)
    model.load_state_dict(ckpt['model_state'])
    print(f'  Best was from: {ckpt["phase"]}  epoch {ckpt["epoch"]}')

    _, test_acc, test_f1, test_preds, test_labels = evaluate(model, test_loader, criterion)
    print(f'\nTEST — acc: {test_acc:.4f}  F1: {test_f1:.4f}')

    report_str = classification_report(
        test_labels, test_preds,
        target_names=[IDX_TO_CLASS[i] for i in range(NUM_CLASSES)],
        zero_division=0
    )
    print(report_str)

    with open(f'{WORKING}/{model_name}_test_report.txt', 'w') as f:
        f.write(f'Model: {model_name}\n')
        f.write(f'Best from  : {ckpt["phase"]}  epoch {ckpt["epoch"]}\n')
        f.write(f'Test Acc   : {test_acc:.6f}\n')
        f.write(f'Test F1    : {test_f1:.6f}\n\n')
        f.write(report_str)

    # ── Learning curves ──
    all_tr  = tr1  + tr2
    all_val = val1 + val2
    all_acc = acc1 + acc2
    all_f1  = f1_1 + f1_2
    ep_range   = range(1, len(all_tr) + 1)
    phase_line = p1_ep + 0.5

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    fig.suptitle(f'{model_name.upper()}  |  Phase boundary at epoch {p1_ep}→{p1_ep+1}', fontsize=11)
    for ax in axes:
        ax.axvline(phase_line, color='gray', linestyle='--', alpha=0.6, label='unfreeze')
    axes[0].plot(ep_range, all_tr,  label='train')
    axes[0].plot(ep_range, all_val, label='val')
    axes[0].set_title('Loss'); axes[0].legend()
    axes[1].plot(ep_range, all_acc, color='tab:green')
    axes[1].set_title('Val Accuracy')
    axes[2].plot(ep_range, all_f1,  color='tab:orange')
    axes[2].set_title('Val F1')
    for ax in axes:
        ax.set_xlabel('Epoch')
    plt.tight_layout()
    plt.savefig(f'{WORKING}/{model_name}_curves.png', dpi=120)
    plt.show()

    return best_val_loss, test_acc, test_f1


In [ ]:



MODELS_TO_RUN = ['mobilenet', 'vgg16', 'vgg19', 'inception']   

import csv, os
WORKING = '/kaggle/working'

results = {}
for model_name in MODELS_TO_RUN:
    best_val_loss, test_acc, test_f1 = train_model(model_name)
    results[model_name] = {'best_val_loss': best_val_loss, 'test_acc': test_acc, 'test_f1': test_f1}

# ── Summary ──
print('\n' + '='*60)
print('FINAL RESULTS (sorted by test F1)')
print('='*60)
print(f"  {'Model':<12}  {'Best Val Loss':<16}  {'Test Acc':<12}  {'Test F1'}")
print('-'*60)
for name, r in sorted(results.items(), key=lambda x: -x[1]['test_f1']):
    print(f"  {name:<12}  {r['best_val_loss']:.4f}            {r['test_acc']:.4f}        {r['test_f1']:.4f}")

# ── Save summary CSV ──
with open(f'{WORKING}/results_summary.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['model','best_val_loss','test_acc','test_f1'])
    writer.writeheader()
    for name, r in sorted(results.items(), key=lambda x: -x[1]['test_f1']):
        writer.writerow({'model': name, **{k: round(v, 6) for k, v in r.items()}})

print(f'\nOutputs saved to {WORKING}/')
print('Files:', sorted(os.listdir(WORKING)))
